## Load the dataset

This should already be filtered for high quality german sentences

In [ ]:
import pandas as pd

dataset = pd.read_json("../data/interim/german_sentences_filtered.jsonl", lines=True)
dataset.head()

# Create Ollama client and Load Env

Load `OLLAMA_API_KEY` variable

In [ ]:
from ollama import Client
import os
from dotenv import load_dotenv

load_dotenv() 

client = Client(
            host="https://ollama.com",
            headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
        )

## Create corruption pair

- Using a bigger LLM `GPT-OSS-120b` we create a corrupted pair for each sentence
- The number of issues and the types are also recorded

In [ ]:
import json
from src.utils import process_in_batches
from src.data.corruptor import corrupt_sentence

all_sentences = dataset["cleaned_sentence"].to_list()

print(f"Starting to corrupt {len(all_sentences)} rows...")

for current_batch in process_in_batches(all_sentences, batch_size=10):
    batch_result = corrupt_sentence(current_batch, client)

    with open("../data/interim/german_sentences_filtered_corrupted2.jsonl", "a", encoding="utf-8") as f:
        for line in batch_result:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")

## We filter the data once mode for Quality

As the last step `GPT-OSS-120b` goes through the pairs again and removes examples that don't have enough value for training or invalid

In [ ]:
df_corrupted = pd.read_json("../data/interim/german_sentences_filtered_corrupted2.jsonl", lines=True)
candidates = list(zip(df_corrupted["original"], df_corrupted["corrupted"]))

In [ ]:
from tqdm import tqdm
from src.data.quality import validate_corruption_pair
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 3


def process_pair(pair):
    clean, corrupt = pair
    evaluation = validate_corruption_pair(clean, corrupt, client)
    return { "input": corrupt, "label": clean, "verdict": evaluation["verdict"], "reason": evaluation["reason"] }


with tqdm(total=len(candidates), desc="Evaluating pairs") as pbar:
    for i in range(0, len(candidates), BATCH_SIZE):
        batch = candidates[i:i + BATCH_SIZE]
        with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
            futures = {executor.submit(process_pair, pair): pair for pair in batch}
            for future in as_completed(futures):
                evaluated_data = future.result()
                with open("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", "a", encoding="utf-8") as f:
                    f.write(json.dumps(evaluated_data, ensure_ascii=False) + "\n")
                pbar.update(1)

# Create the final dataset ready for training

In [ ]:
df_eval = pd.read_json("../data/interim/german_sentences_filtered_corrupted_evaluated.jsonl", lines=True)

df_final = df_eval[df_eval["verdict"] == "KEEP"].copy()
df_final.rename(columns={"input": "corrupted", "label": "original"}, inplace=True)
df_final.to_json("../data/processed/german_sentences_corrupted_final.jsonl", orient="records", lines=True)